# Run This API on Google Colab

> Free option for a low-capacity laptop: the API runs on Colab's machine and gets a public URL through Cloudflare Tunnel.

This notebook will:
- optionally mount Google Drive
- clone the repository
- install system and Python dependencies
- start the FastAPI server
- expose it with a temporary public URL
- verify the `/health` endpoint

Important notes:
- Colab sessions are temporary and can stop when idle.
- The public URL changes every time you restart the tunnel.
- This is suitable for demos, testing, and short-term sharing.

In [ ]:
# Optional: mount Google Drive if you want to save files or work from Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone The Repository

> If the repository is public, this cell is enough. If it is private, upload a zip to Drive or use a GitHub token-based clone.

In [ ]:
!rm -rf /content/multiagent
!git clone https://github.com/DinethiWijesinghe/multiagent.git /content/multiagent
%cd /content/multiagent

# Install Dependencies

> This installs Linux packages needed by OCR and the Python packages needed by the FastAPI server.

In [ ]:
%cd /content/multiagent
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr libgl1 libglib2.0-0 wget
!python -m pip install --upgrade pip
!pip install fastapi uvicorn python-multipart numpy pillow opencv-python scikit-learn pytesseract requests

# Start The FastAPI Server

> This runs the backend on Colab at port `8000`.

In [ ]:
import subprocess
import time
import requests

%cd /content/multiagent

server_process = subprocess.Popen(
    [
        "uvicorn",
        "multiagent.api_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ]
)

time.sleep(8)

try:
    health = requests.get("http://127.0.0.1:8000/health", timeout=10)
    print("Local health:", health.status_code)
    print(health.json())
except Exception as exc:
    print("Health check failed:", exc)

# Create A Public URL

> Cloudflare Tunnel gives you a temporary public URL without using your laptop.

In [ ]:
import re
import subprocess
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
 )

public_url = None
start = time.time()

while time.time() - start < 30:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\nPublic URL:", public_url if public_url else "Not found yet")

# Test The Public API

> If the previous cell printed a URL, this verifies that the API is reachable from the internet.

In [ ]:
import requests

if public_url:
    response = requests.get(f"{public_url}/health", timeout=20)
    print("Public health:", response.status_code)
    print(response.json())
else:
    print("No public URL detected. Re-run the tunnel cell.")